# 00 · Setup, preflight and seed

This lab is the *claims / COA knowledge-graph assistant* exercise, rebuilt to run **on your laptop**. Nothing runs on EC2: the notebooks import the lab's `src/` code directly and talk to AWS with **your** credentials.

| Component | Where it runs | Used by |
|---|---|---|
| `src/` code, MCP server, evidence scripts | your laptop (these notebooks) | all goals |
| Neo4j 5.x (graph + metric vector index) | your laptop, local install | goals 3, 4, 5 |
| DynamoDB `claims-sessions` (Agent A memory) | AWS | goal 1 |
| IAM `<prefix>-McpVectorRole` | AWS | goal 2 |
| OpenSearch Serverless collection (BM25 + kNN, vector store A) | AWS | goals 3, 4 |
| Aurora PostgreSQL Serverless v2 + pgvector (vector store B, metric source of record) | AWS (or a local PostgreSQL) | goals 3, 4 |
| Bedrock: Titan Text Embeddings V2, Nova Pro | AWS | everything that embeds or answers |
| S3 evidence bucket | AWS | **grading** |

**Grading works on evidence, not on your environment.** A goal is finished only when its `emit_evidence` step has run and the JSON file is uploaded to `s3://$EVIDENCE_BUCKET/evidence/`.

**Do the goals in order.** Goal 4's evidence depends on the data seeded in goal 3.

## Step 1 · Prerequisites (once)

1. **Python 3.10+ environment** in the project root:
   ```bash
   python -m venv .venv && source .venv/bin/activate
   pip install -r requirements.txt
   python -m ipykernel install --user --name gmlab   # then pick the 'gmlab' kernel for these notebooks
   ```
2. **AWS credentials** for a sandbox account (admin), e.g. `aws configure` or `export AWS_PROFILE=...`. Check with `aws sts get-caller-identity`.
3. **Bedrock model access** in your Region for *Titan Text Embeddings V2* and *Amazon Nova Pro* (Bedrock console → Model access).
4. **Neo4j 5.13 or later** running locally (Neo4j Desktop or a package install), with the bolt port reachable (default `bolt://localhost:7687`) and a password you know. The lab needs vector indexes (`db.index.vector.queryNodes`) and `COUNT {}` subqueries, so it needs a 5.x server.

## Step 2 · Create the AWS resources (once)

```bash
curl -s https://checkip.amazonaws.com                       # your public IP
terraform -chdir=infra/terraform init
terraform -chdir=infra/terraform apply -var my_ip_cidr=<your.ip>/32
```

The apply takes about 10-15 minutes: Aurora and the OpenSearch Serverless collection are the slow parts.

To use a **local PostgreSQL** (with the `vector` extension available) instead of Aurora, add `-var create_aurora=false`.

## Step 3 · Write `.env`

```bash
python scripts/make_env.py                  # asks for your Neo4j password
python scripts/make_env.py --local-postgres # if you skipped Aurora
```

`.env` holds passwords: it's in `.gitignore`. Never commit it.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))   # this folder (notebooks/) holds labkit.py
import labkit

for key in ['AWS_REGION', 'SESSION_TABLE', 'EVIDENCE_BUCKET', 'MCP_VECTOR_ROLE_NAME', 'OPENSEARCH_HOST', 'OPENSEARCH_SERVICE',
            'NEO4J_URI', 'PGHOST', 'PGSSLMODE', 'EMBEDDING_DIMENSIONS']:
    print(f'{key:22} {os.environ.get(key, "(not set)")}')
print('NEO4J_PASSWORD / PGPASSWORD set:', bool(os.environ.get('NEO4J_PASSWORD')), bool(os.environ.get('PGPASSWORD')))
print('project root:', os.getcwd())

## Step 4 · Preflight: AWS identity and Bedrock

`src/preflight.py` checks your credentials, makes one Titan embedding and calls Nova Pro through the Converse API.

In [ ]:
!{sys.executable} -m src.preflight

## Step 5 · Connectivity to the three stores

Each probe is independent, so one failure doesn't hide the others.

In [ ]:
from src.graph_client import GraphLabDriver
from src.opensearch_client import OpenSearchVectorStore
from src.pgvector_client import PgVectorStore

try:
    lab = GraphLabDriver(connect_retries=3)
    with lab.driver.session() as s:
        v = s.run('CALL dbms.components() YIELD name, versions RETURN name, versions[0] AS version').single()
    print('[OK]  Neo4j', v['name'], v['version'])
    lab.close()
except Exception as e:
    print('[FAIL] Neo4j:', e)

try:
    os_store = OpenSearchVectorStore()
    os_store.wait_until_ready(retries=3)
    print('[OK]  OpenSearch', os.environ.get('OPENSEARCH_SERVICE'), os.environ.get('OPENSEARCH_HOST'))
except Exception as e:
    print('[FAIL] OpenSearch:', e)

try:
    pg = PgVectorStore()
    with pg.conn.cursor() as cur:
        cur.execute('SELECT version()')
        print('[OK]  PostgreSQL', cur.fetchone()[0].split(',')[0])
        cur.execute("SELECT default_version, installed_version FROM pg_available_extensions WHERE name = 'vector'")
        row = cur.fetchone()
        print('      pgvector available:', row[0] if row else 'NO - install the vector extension',
              '| installed:', (row[1] if row else None) or 'not yet (seed creates it)')
    pg.close()
except Exception as e:
    print('[FAIL] PostgreSQL:', e)

| Failure | Usual cause |
|---|---|
| OpenSearch `auth error (403)` | Your role isn't in the collection's data access policy: re-run `terraform apply` with the same credentials you use here |
| PostgreSQL timeout | Your public IP changed: re-apply with the new `my_ip_cidr`. The first connection after Aurora auto-pauses can take ~30 s |
| Neo4j `Unable to retrieve routing information` / auth | Neo4j isn't running, or `NEO4J_URI` / `NEO4J_PASSWORD` in `.env` is wrong |
| pgvector `NO` | Local PostgreSQL without pgvector: install it (`CREATE EXTENSION vector` must work) |

## Step 6 · Seed every store (the lab's deploy step)

`python -m src.main --seed-only` does the following:
- initialises the Neo4j constraints and vector index, the OpenSearch k-NN index and the pgvector tables;
- seeds the governed-metric source of record (`quarterly_capital` = 1.85, `cargo_experience` = 0.62);
- writes the demo conversation `s-demo` (turns 1, 2, 3) to DynamoDB;
- ingests the commercial-property ontology and its metric, and dual-ingests one document chunk (Nexus Tower).

It is write-only and defect-independent, so it succeeds on the broken start state. OpenSearch Serverless is eventually consistent, so the last step waits about a minute until the chunk is searchable.

In [ ]:
!{sys.executable} -m src.main --seed-only

## Step 7 · Preview the evidence workflow

`preview` reads live state without touching any graded evidence. Expect:
- 1 document in each vector store;
- 2 ontology classes;
- 1 governed metric;
- 1 entity;
- **0** `INSTANCE_OF` links. That last one is the defect goal 3 repairs.

In [ ]:
!{sys.executable} -m src.emit_evidence preview

You're ready. Continue with **01 · Memory tools and turn order**.

At any point, `python scripts/check_evidence.py` (or `labkit.grade()`) shows what the grader would see in your evidence bucket.